# 01 — Reward Model Training

Trains the reward model that will drive RLOO in notebook 02. The base model is
`Qwen/Qwen2.5-1.5B-Instruct` with a scalar regression head; we add LoRA adapters via
TRL's `RewardTrainer` (TRL 1.x applies PEFT internally when you pass `peft_config`).

**Output:** `outputs/reward_model/` (LoRA adapter + tokenizer + metrics).

In [ ]:
# Run once in a fresh environment:
# !pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('Working dir:', Path.cwd())

In [ ]:
import json
import torch
from peft import TaskType
from trl import RewardConfig, RewardTrainer

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.utils.device import runtime_profile, adapt_quant_cfg
from src.data.preferences import load_hh_rlhf_for_reward_model
from src.models.reward import load_base_reward_model, make_lora_config

profile = runtime_profile()
cfg = load_config('configs/config.yaml')
seed_everything(cfg['seed'])
# Disable 4-bit quant on non-CUDA machines so the notebook at least loads locally.
cfg['quantization'] = adapt_quant_cfg(cfg['quantization'], profile)
print(json.dumps(cfg['reward_model'], indent=2))

## 1. Load HH-RLHF preference pairs

Our loader splits each transcript into `(prompt, chosen, rejected)` triples where
`chosen`/`rejected` already contain the full text (prompt + response). The TRL
`RewardTrainer` accepts that implicit-prompt format directly.

In [ ]:
train_ds, eval_ds = load_hh_rlhf_for_reward_model(
    num_train=cfg['reward_model']['num_train_samples'],
    num_eval=cfg['reward_model']['num_eval_samples'],
    seed=cfg['seed'],
)
# RewardTrainer wants only the chosen/rejected columns in implicit-prompt mode.
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in ('chosen', 'rejected')])
eval_ds  = eval_ds.remove_columns([c for c in eval_ds.column_names  if c not in ('chosen', 'rejected')])
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')
print('--- sample ---')
print('CHOSEN:  ', train_ds[0]['chosen'][:300], '...')
print('REJECTED:', train_ds[0]['rejected'][:300], '...')

## 2. Build the base reward model (no PEFT yet)

TRL 1.x wraps the model with LoRA itself when given a `peft_config`, so we just hand
it the bare 4-bit-quantized backbone.

In [ ]:
model, tokenizer = load_base_reward_model(
    model_name=cfg['base_model'],
    quant_cfg=cfg['quantization'],
)
peft_config = make_lora_config(cfg['lora'], task_type=TaskType.SEQ_CLS)
print('peft_config:', peft_config)

## 3. Train (Bradley–Terry loss via `RewardTrainer`)

In [ ]:
rm_cfg = cfg['reward_model']
output_dir = cfg['paths']['reward_model_dir']
Path(output_dir).mkdir(parents=True, exist_ok=True)

reward_config = RewardConfig(
    output_dir=output_dir,
    num_train_epochs=rm_cfg['num_train_epochs'],
    per_device_train_batch_size=rm_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=rm_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=rm_cfg['gradient_accumulation_steps'],
    learning_rate=rm_cfg['learning_rate'],
    warmup_ratio=rm_cfg['warmup_ratio'],
    logging_steps=rm_cfg['logging_steps'],
    eval_strategy='steps',
    eval_steps=rm_cfg['eval_steps'],
    save_steps=rm_cfg['save_steps'],
    save_total_limit=2,
    bf16=profile.use_bf16,
    gradient_checkpointing=profile.has_cuda,
    max_length=rm_cfg['max_length'],
    report_to='none',
    seed=cfg['seed'],
)

trainer = RewardTrainer(
    model=model,
    args=reward_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

train_result = trainer.train()
print(train_result.metrics)

## 4. Save adapter + tokenizer + metrics

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

metrics = trainer.evaluate()
with open(Path(output_dir) / 'final_metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2)
print('Saved to', output_dir)
print('Eval metrics:', metrics)

## 5. Sanity check — does the reward model prefer `chosen` over `rejected`?

On a small held-out batch we expect the margin (r_chosen − r_rejected) to be
positive on most examples. Headline accuracy is reported by `RewardTrainer` above.

In [ ]:
import numpy as np

trainer.model.eval()
device = next(trainer.model.parameters()).device
n_check = 20
margins = []
with torch.no_grad():
    for row in eval_ds.select(range(min(n_check, len(eval_ds)))):
        toks_c = tokenizer(row['chosen'],   truncation=True, max_length=rm_cfg['max_length'], return_tensors='pt').to(device)
        toks_r = tokenizer(row['rejected'], truncation=True, max_length=rm_cfg['max_length'], return_tensors='pt').to(device)
        r_c = trainer.model(**toks_c).logits.squeeze().item()
        r_r = trainer.model(**toks_r).logits.squeeze().item()
        margins.append(r_c - r_r)

margins = np.array(margins)
print(f'mean margin: {margins.mean():+.3f}  | sign agreement: {(margins > 0).mean():.1%}')